# 3 Auswertung auf den Ausschnitten

Die Punkte werden in der vorhergesagten Heatmap gesucht (lokale Maxima, dann Subpixel über eine Parabel durch die Logarithmen der Nachbarpixel, siehe `grenzpunkte.py`) und mit den Labels verglichen.

1. Precision, Recall und Lageabweichung pro Schwelle
2. Vergleich mit dem Abstand der beiden menschlichen Digitalisierungen D1 ↔ D2
3. Bildbeispiele: Fehldetektionen, verpasste Punkte und Versatz pro Plan

Ausgewertet wird der val-Split, der test-Split erst ganz am Schluss.

In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from torchvision.io import decode_image

from grenzpunkte import (BATCH_SIZE, DATA, DATENSATZ, MAX_ABSTAND, PunktDataset, detektieren, device, modell_laden,
                         zuordnen)

SPLIT = "val"
SCHWELLEN = [0.1, 0.3, 0.5, 0.7, 0.9]  # Mindestwert der Heatmap für eine Detektion
SCHWELLE_BEISPIELE = 0.9
AUSGABE = "data/auswertung"
random.seed(0)

## Heatmaps vorhersagen und Punkte suchen
Gesucht wird mit der tiefsten Schwelle, höhere Schwellen filtern danach nur noch.

In [ ]:
dataset = PunktDataset(SPLIT)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, num_workers=4)  # ohne shuffle: Reihenfolge = dataset.bilder
model = modell_laden()

detektionen = []
with torch.no_grad():
    for X, _ in loader:
        heatmaps = torch.sigmoid(model(X.to(device)))[:, 0].cpu().numpy()
        detektionen += [detektieren(hm, min(SCHWELLEN)) for hm in heatmaps]
print(f"{len(dataset)} Ausschnitte, {sum(len(d) for d in detektionen)} Detektionen")

## Precision, Recall und Lage pro Schwelle
Detektion und Label bilden ein Paar, wenn sie höchstens 8 px (600 DPI) auseinander liegen (eins zu eins, kleinste Summe der Abstände). Die Lage wird über die Pixelgrösse des Plans in cm umgerechnet.
Die Tabelle wird für `06_grafiken.ipynb` gespeichert.

In [ ]:
plaene = pd.read_csv(f"{DATENSATZ}/plaene.csv").query("status == 'ok'").set_index("plan_id")
meter_pro_px = plaene["bodenpixel_m"] * plaene["scan_dpi"] / 600  # Ausschnitte sind 600 DPI

tabelle = []
for schwelle in SCHWELLEN:
    tp = fp = fn = 0
    fehler_cm = []
    for bild, det in zip(dataset.bilder, detektionen):
        det = det[det[:, 2] >= schwelle, :2]
        labels = dataset.punkte.get(bild, np.empty((0, 2)))
        abstaende = [abstand for _, _, abstand in zuordnen(det, labels)]
        tp, fp, fn = tp + len(abstaende), fp + len(det) - len(abstaende), fn + len(labels) - len(abstaende)
        fehler_cm += [a * meter_pro_px[bild.rsplit("_", 1)[0]] * 100 for a in abstaende]
    tabelle.append({"schwelle": schwelle, "tp": tp, "fp": fp, "fn": fn,
                    "precision": tp / (tp + fp), "recall": tp / (tp + fn),
                    "median_cm": np.median(fehler_cm), "p95_cm": np.percentile(fehler_cm, 95),
                    "anteil_10cm": np.mean(np.array(fehler_cm) <= 10)})
tabelle = pd.DataFrame(tabelle)
tabelle.to_csv(f"{AUSGABE}/stufe1_schwellen_{SPLIT}.csv", index=False)
tabelle.round(3)

## Vergleich: Abstand der beiden Digitalisierungen D1 ↔ D2
Auf denselben Plänen. Eine einzelne Digitalisierung liegt etwa halb so weit vom Label entfernt, weil das Label das Mittel aus D1 und D2 ist.

In [ ]:
d12_cm = np.concatenate([pd.read_csv(f"{DATENSATZ}/labels/{p}.csv")["d12_px"].dropna() * plaene.at[p, "bodenpixel_m"] * 100
                         for p in {bild.rsplit("_", 1)[0] for bild in dataset.bilder}])
print(f"D1<->D2: Median {np.median(d12_cm):.1f} cm, p95 {np.percentile(d12_cm, 95):.1f} cm, "
      f"<= 10 cm {(d12_cm <= 10).mean():.1%}")

## Bildbeispiele
Stark vergrössert, bei Schwelle 0.9: Label = grünes +, Detektion = rotes ×. Die Bilder werden auch in `data/auswertung/` gespeichert.

In [ ]:
def galerie(beispiele, titel, datei, zoom):
    """Zeichnet pro Beispiel einen Ausschnitt von +-zoom Pixeln um (x, y).
    beispiele: Liste von (bild, x, y, text, labels, detektionen)."""
    spalten = 6
    zeilen = int(np.ceil(len(beispiele) / spalten))
    fig, axes = plt.subplots(zeilen, spalten, figsize=(3 * spalten, 3.3 * zeilen), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, (bild, x, y, text, labels, det) in zip(axes.flat, beispiele):
        image = decode_image(f"{DATA}/bilder/{bild}.png").permute(1, 2, 0).numpy()
        ax.imshow(image, extent=(0, 256, 256, 0), interpolation="nearest")  # Pixelmitte liegt so bei i + 0.5
        ax.scatter(labels[:, 0], labels[:, 1], marker="+", c="lime", s=300, linewidths=1)
        ax.scatter(det[:, 0], det[:, 1], marker="x", c="red", s=200, linewidths=1)
        ax.set_xlim(x - zoom, x + zoom)
        ax.set_ylim(y + zoom, y - zoom)
        ax.set_title(text, fontsize=8)
    fig.suptitle(titel)
    fig.tight_layout()
    fig.savefig(datei, dpi=100)
    plt.show()

In [ ]:
# Fehldetektionen, verpasste Punkte und Treffer bei SCHWELLE_BEISPIELE sammeln
fehl, verpasst, treffer = [], [], []
for bild, det in zip(dataset.bilder, detektionen):
    det = det[det[:, 2] >= SCHWELLE_BEISPIELE, :2]
    labels = dataset.punkte.get(bild, np.empty((0, 2)))
    paare = zuordnen(det, labels)
    fehl += [(bild, x, y, bild, labels, det) for i, (x, y) in enumerate(det) if i not in {p[0] for p in paare}]
    verpasst += [(bild, x, y, bild, labels, det) for j, (x, y) in enumerate(labels) if j not in {p[1] for p in paare}]
    for i, j, _ in paare:
        dx, dy = det[i] - labels[j]
        treffer.append({"bild": bild, "plan": bild.rsplit("_", 1)[0], "x": labels[j][0], "y": labels[j][1],
                        "dx": dx, "dy": dy, "labels": labels, "det": det})
print(f"{len(fehl)} Fehldetektionen, {len(verpasst)} verpasste Punkte, {len(treffer)} Treffer")

In [ ]:
galerie(random.sample(fehl, 24), f"Fehldetektionen (zufällige 24 von {len(fehl)})", f"{AUSGABE}/fehldetektionen.png", 40)
galerie(random.sample(verpasst, 24), f"Verpasste Punkte (zufällige 24 von {len(verpasst)})",
        f"{AUSGABE}/verpasste_punkte.png", 40)

### Versatz pro Plan
Mittlerer Versatz Detektion − Label pro Plan: die drei Pläne mit dem grössten und der mit dem kleinsten Versatz, je sechs Treffer. Ein gemeinsamer Versatz aller Punkte eines Plans deutet auf einen systematischen Fehler der Labels hin.

In [ ]:
treffer = pd.DataFrame(treffer)
versatz = treffer.groupby("plan")[["dx", "dy"]].mean()
versatz["betrag"] = np.hypot(versatz["dx"], versatz["dy"])
versatz = versatz.sort_values("betrag")
beispiele = []
for plan in list(versatz.index[-3:][::-1]) + [versatz.index[0]]:
    dx, dy = versatz.loc[plan, ["dx", "dy"]]
    for _, t in treffer[treffer["plan"] == plan].sample(6, random_state=0).iterrows():
        text = f"{plan}: Plan-Versatz ({dx:+.2f}, {dy:+.2f}) px\nhier ({t['dx']:+.2f}, {t['dy']:+.2f}) px"
        beispiele.append((t["bild"], t["x"], t["y"], text, t["labels"], t["det"]))
galerie(beispiele, "Treffer: 3 Pläne mit grösstem, 1 mit kleinstem mittlerem Versatz (Detektion − Label)",
        f"{AUSGABE}/versatz.png", 12)